In [1]:
import sys 
import re
from pathlib import Path 
sys.path.append(str(Path().resolve().parents[0]))

import pandas as pd

from config import RAW_DATA_DIR, PROCESSED_DATA_DIR

In [2]:
pokemon_df = pd.read_parquet(PROCESSED_DATA_DIR / "pokemon_data_features.parquet")
evolution_df = pd.read_csv(RAW_DATA_DIR / "pokemon_evolutions.csv")

In [3]:
print(pokemon_df.shape)
print(evolution_df.shape)

(1045, 51)
(1179, 47)


In [4]:
def normalize_name(name: str) -> str:
    """Normalize Pokemon names to ensure proper matching"""
    name = name.strip().lower()
    # Replace spaces with hyphens
    name = name.replace(" ", "-")
    # Handle gender symbols
    name = name.replace("♀", "-f").replace("♂", "-m")
    # Remove special characters (keep letters, numbers, hyphens)
    name = re.sub(r"[^a-z0-9\-]", "", name)
    return name


def classify_special_pokemon(name: str) -> str:
    """Convert special cases of Pokemon into their base species name."""

    # Handle remaining manual special cases first
    name_map = {
        "dusk-mane-necrozma": "necrozma",
        "dawn-wings-necrozma": "necrozma",
        "ultra-necrozma": "necrozma",
        "eternatus-eternamax": "eternatus",
        "ash-greninja": "greninja",
        "own-tempo-rockruff": "rockruff",
        "partner-pikachu": "pikachu",
        "partner-eevee": "eevee",
        "hoopa-hoopa-unbound": "hoopa",
        "hoopa-hoopa-confined": "hoopa",
        "hoopa-unbound": "hoopa",
        "hoopa-confined": "hoopa",
        "keldeo-ordinary-form": "keldeo",
        "keldeo-resolute-form": "keldeo",
        "no-drive-genesect": "genesect",
        "douse-drive-genesect": "genesect",
        "shock-drive-genesect": "genesect",
        "burn-drive-genesect": "genesect",
        "chill-drive-genesect": "genesect",
        "galarian-darmanitan": "galarian-darmanitan",
        "galarian-darmanitan-zen-mode": "galarian-darmanitan",
        "castform-normal": "castform",
        "castform-sunny-form": "castform",
        "castform-rainy-form": "castform",
        "castform-snowy-form": "castform",
        "burmy-plant-cloak": "burmy",
        "burmy-sandy-cloak": "burmy",
        "burmy-trash-cloak": "burmy",
        "cherrim-overcast-form": "cherrim",
        "cherrim-sunshine-form": "cherrim",
        "shellos-west-sea": "shellos",
        "shellos-east-sea": "shellos",
        "gastrodon-west-sea": "gastrodon",
        "gastrodon-east-sea": "gastrodon",
        "deerling-spring-form": "deerling",
        "deerling-summer-form": "deerling",
        "deerling-autumn-form": "deerling",
        "deerling-winter-form": "deerling",
        "sawsbuck-spring-form": "sawsbuck",
        "sawsbuck-summer-form": "sawsbuck",
        "sawsbuck-autumn-form": "sawsbuck",
        "sawsbuck-winter-form": "sawsbuck",
        "vivillon-meadow-pattern": "vivillon",
        "floette-red-flower": "floette",
        "floette-yellow-flower": "floette",
        "floette-orange-flower": "floette",
        "floette-blue-flower": "floette",
        "floette-white-flower": "floette",
        "florges-red-flower": "florges",
        "florges-yellow-flower": "florges",
        "florges-orange-flower": "florges",
        "florges-blue-flower": "florges",
        "florges-white-flower": "florges",
        "furfrou-natural-form": "furfrou",
        "xerneas-neutral-mode": "xerneas",
        "xerneas-active-mode": "xerneas",
        "silvally-type-normal": "silvally",
        "minior-meteor-form": "minior",
        "minior-red-core": "minior",
        "minior-core-form": "minior",
        "mimikyu-disguised-form": "mimikyu",
        "mimikyu-busted-form": "mimikyu",
        "alcremie-vanilla-cream": "alcremie",
        "unown-one-form": "unown",
        "flabb": "flabebe",
        "flabb-red-flower": "flabebe",
        "flabb-yellow-flower": "flabebe",
        "flabb-orange-flower": "flabebe",
        "flabb-blue-flower": "flabebe",
        "flabb-white-flower": "flabebe",
        "galarian-darmanitan-standard-mode": "galarian-darmanitan",

    }

    if name in name_map:
        return name_map[name]

    # Mega Pokemon
    if name.startswith("mega-"):
        name = name.replace("mega-", "", 1)

        # Special cases of Mega Pokemon (e.g., mega-charizard-x)
        if name.endswith("-x") or name.endswith("-y"):
            name = name.rsplit("-", 1)[0]

    # Primal form Pokemon
    if name.startswith("primal-"):
        name = name.replace("primal-", "", 1)

    # Handle size variants (e.g., pumpkaboo-small-size)
    size_variants = {"-small-size", "-average-size", "-large-size", "-super-size"}
    for s in size_variants:
        if name.endswith(s):
            name = name.replace(s, "")
            break

    # Handle forme variants (e.g., keldeo-ordinary-forme)
    forme_variants = {"-ordinary-forme", "-resolute-forme"}
    for f in forme_variants:
        if name.endswith(f):
            name = name.replace(f, "")
            break

    return name

In [5]:
evolution_df = evolution_df[["Name", "IsLegendary", "IsMythical", "IsUltraBeast", "EvoStage"]]
print(evolution_df.columns)

Index(['Name', 'IsLegendary', 'IsMythical', 'IsUltraBeast', 'EvoStage'], dtype='str')


In [6]:
evolution_df = evolution_df.rename(columns={"Name": "name"})
print(evolution_df.columns)

Index(['name', 'IsLegendary', 'IsMythical', 'IsUltraBeast', 'EvoStage'], dtype='str')


In [7]:
pokemon_df["name"] = pokemon_df["name"].apply(normalize_name)
evolution_df["name"] = evolution_df["name"].apply(normalize_name)

In [8]:
dataset_names = pokemon_df["name"].to_list()
evolution_names = evolution_df["name"].to_list()

# Find how many pokemon names, the datasets have in common
intersection = (set(dataset_names) & set(evolution_names))
print(len(intersection))

955


In [9]:
# Find out what Pokemon in the original dataset do not have an evolution stage
print(set(dataset_names) - set(intersection))

{'gourgeist-super-size', 'mega-diancie', 'mega-pidgeot', 'shellos', 'mega-tyranitar', 'mega-gyarados', 'gastrodon', 'minior-core-form', 'gourgeist-average-size', 'mega-aggron', 'mega-charizard-y', 'mega-audino', 'alcremie', 'mega-ampharos', 'mega-metagross', 'mega-absol', 'sawsbuck', 'mega-mewtwo-x', 'keldeo-resolute-forme', 'eternatus-eternamax', 'mega-gengar', 'mega-mawile', 'mega-gallade', 'mega-steelix', 'mega-lucario', 'mega-medicham', 'keldeo-ordinary-forme', 'mega-alakazam', 'florges', 'genesect', 'vivillon', 'pumpkaboo-large-size', 'mega-pinsir', 'mega-sceptile', 'mega-rayquaza', 'ash-greninja', 'galarian-darmanitan-standard-mode', 'mega-aerodactyl', 'castform', 'primal-kyogre', 'mega-sharpedo', 'mega-slowbro', 'hoopa-hoopa-unbound', 'mega-banette', 'hoopa-hoopa-confined', 'ultra-necrozma', 'flabb', 'mega-glalie', 'pumpkaboo-small-size', 'mega-abomasnow', 'mimikyu', 'partner-pikachu', 'mega-blaziken', 'burmy', 'mega-salamence', 'mega-camerupt', 'own-tempo-rockruff', 'deerling',

In [10]:
pokemon_df["base_name"] = pokemon_df["name"].apply(classify_special_pokemon)
evolution_df["base_name"] = evolution_df["name"].apply(classify_special_pokemon)
evolution_df = evolution_df.drop_duplicates(subset=["base_name", "EvoStage"])

In [11]:
evolution_df = evolution_df[["base_name", "EvoStage"]]

pokemon_df = pd.merge(
    pokemon_df,
    evolution_df,
    on="base_name",
    how="left"
)

In [12]:
print(pokemon_df.head(5))
print(pokemon_df.isna().sum())

            name  total_points  generation  log_height_m  log_weight_kg  \
0      bulbasaur           318           1      0.530628       2.066863   
1        ivysaur           405           1      0.693147       2.639057   
2       venusaur           525           1      1.098612       4.615121   
3  mega-venusaur           625           1      1.223775       5.053056   
4     charmander           309           1      0.470004       2.251292   

   is_mega  is_primal  is_form  status_mythical  status_normal  ...  \
0        0          0        0            False           True  ...   
1        0          0        0            False           True  ...   
2        0          0        0            False           True  ...   
3        1          0        0            False           True  ...   
4        0          0        0            False           True  ...   

   type_2_rock  type_2_steel  type_2_water  growth_rate_fast  \
0        False         False         False             Fal

In [13]:
pokemon_df = pokemon_df.rename(columns={"EvoStage": "evo_stage"})
pokemon_df["evo_stage"] = pokemon_df["evo_stage"].astype(int)
pokemon_df = pokemon_df.drop(columns=["name", "base_name"])

pokemon_df.to_parquet(PROCESSED_DATA_DIR / "pokemon_data_features_evo_stage.parquet", index=False)